# The Lahore Canyon: narrated video (GIS Day 2026)

**Muhammad Mehdi & Hussain Ahmad**, BS Geography, 7th Semester (Morning), Institute of Geography, University of the Punjab, Lahore

This notebook builds a narrated explainer video of the app:

- 4K screenshots of the app (red circles mark what is being explained), downloaded from the project's GitHub repository
- smooth camera moves that zoom onto each circled detail while it is explained
- a male AI voice (Microsoft Edge neural voice, via `edge-tts`)
- subtitles burned into the video, plus a separate `.srt` file

**How to run:** open in Google Colab, then *Runtime → Run all*. It takes about 10–15 minutes. The finished files are `lahore_canyon_explainer.mp4` and `lahore_canyon_explainer.srt`.

To change the voice, edit `VOICE` in the settings cell. To change what is said, edit the `SCENES` cell.

In [ ]:
# Install the voice library (Colab already has ffmpeg, Pillow and numpy)
!pip -q install edge-tts

In [ ]:
# ---------------- Settings ----------------
VOICE = "en-US-AndrewNeural"          # male voices: en-US-AndrewNeural, en-US-GuyNeural, en-GB-RyanNeural, en-US-ChristopherNeural
VOICE_FALLBACKS = ["en-US-GuyNeural", "en-GB-RyanNeural"]
RATE = "-4%"                          # speaking speed: "-10%" slower, "+5%" faster
TTS_ENGINE = "edge"                   # "edge" (internet needed) or "silent" (test the visuals without a voice)

W, H, FPS = 1920, 1080, 30            # output size and frame rate
REPO = "https://raw.githubusercontent.com/MUHAMMAD-MEHDI12/lahore-canyon/main/media/"
IMAGES = ["intro", "heat_walled", "heat_dha", "photo", "sun", "building", "smog"]
WORK = "lahore_video_work"
OUT_MP4 = "lahore_canyon_explainer.mp4"
OUT_SRT = "lahore_canyon_explainer.srt"
LOCAL_IMAGES = None                   # set to a folder path to use local screenshots instead of downloading

In [ ]:
import os, io, re, json, math, wave, shutil, subprocess, asyncio, urllib.request
import numpy as np
from PIL import Image, ImageDraw, ImageFont

os.makedirs(WORK, exist_ok=True)
os.makedirs(f"{WORK}/img", exist_ok=True)
os.makedirs(f"{WORK}/tts", exist_ok=True)
os.makedirs(f"{WORK}/scenes", exist_ok=True)

# screenshots
for name in IMAGES:
    dst = f"{WORK}/img/{name}.jpg"
    if os.path.exists(dst):
        continue
    if LOCAL_IMAGES:
        shutil.copy(os.path.join(LOCAL_IMAGES, name + ".jpg"), dst)
    else:
        urllib.request.urlretrieve(REPO + name + ".jpg", dst)
print("screenshots:", sorted(os.listdir(f"{WORK}/img")))

# fonts: Inter from Google Fonts, falling back to DejaVu Sans
FONT_FILE = f"{WORK}/Inter.ttf"
try:
    if not os.path.exists(FONT_FILE):
        urllib.request.urlretrieve("https://raw.githubusercontent.com/google/fonts/main/ofl/inter/Inter%5Bopsz,wght%5D.ttf", FONT_FILE)
    ImageFont.truetype(FONT_FILE, 20)
except Exception as e:
    print("Inter not available, using DejaVu Sans:", e)
    FONT_FILE = None

def font(size, weight="Regular"):
    if FONT_FILE:
        f = ImageFont.truetype(FONT_FILE, size)
        try:
            f.set_variation_by_name(weight)
        except Exception:
            pass
        return f
    bold = weight in ("SemiBold", "Bold", "ExtraBold")
    for p in ["/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"]:
        if os.path.exists(p):
            return ImageFont.truetype(p, size)
    return ImageFont.load_default()

## The script

Each scene has a picture (an app screenshot or a title card), a chapter title, and a list of narration lines.
Each line names a **focus**: the part of the screenshot the camera moves to while that line is spoken.
Focus points are `(centre x, centre y, view width)` in 1920 × 1080 screenshot coordinates.

In [ ]:
FOCUS = {
    "intro":       {"full": (960, 540, 1920), "tag": (420, 560, 1100), "buttons": (420, 700, 1100)},
    "heat_walled": {"full": (960, 540, 1920), "map": (760, 500, 1350), "hot": (790, 390, 900),
                    "cool": (590, 700, 900), "num": (1640, 610, 900)},
    "heat_dha":    {"full": (960, 540, 1920), "map": (760, 500, 1350), "typical": (1640, 610, 900),
                    "green": (1640, 860, 900)},
    "photo":       {"full": (960, 540, 1920), "map": (760, 500, 1350), "layer": (1640, 260, 900),
                    "green": (600, 720, 900)},
    "sun":         {"full": (960, 540, 1920), "map": (760, 480, 1350), "clock": (420, 900, 820)},
    "building":    {"full": (960, 540, 1920), "bld": (720, 470, 1000), "chart": (1640, 430, 900)},
    "smog":        {"full": (960, 540, 1920), "map": (760, 520, 1450), "label": (1290, 240, 1100),
                    "lid": (560, 900, 820), "mixing": (1000, 900, 820)},
}

SCENES = [
    {"card": "title", "chapter": "The Lahore Canyon", "lines": [
        ("Welcome. This is The Lahore Canyon, a GIS Day 2026 project by Muhammad Mehdi and Hussain Ahmad.", None),
        ("We are seventh-semester BS Geography students at the Institute of Geography, University of the Punjab, Lahore.", None),
        ("This year's GIS Day theme is GIS: Powering Purpose and Innovation.", None),
    ]},
    {"img": "intro", "chapter": "The application", "lines": [
        ("The Lahore Canyon is a three-dimensional web map of every building in two parts of Lahore.", "full"),
        ("It compares the old Walled City with the planned neighbourhood of DHA Phase 2.", "full"),
        ("Everything in it is built from free, open satellite data and weather records.", "full"),
        ("Its message is simple: heat has an address, and GIS finds it.", "tag"),
        ("Visitors can follow a guided walkthrough, or go straight to the map and explore it themselves.", "buttons"),
    ]},
    {"card": "data", "chapter": "Data and method", "lines": [
        ("Before we look at the results, here is where every number comes from.", None),
        ("Building footprints and heights come from Google Open Buildings, which estimates heights from Sentinel-2 satellite images.", None),
        ("Ground temperature was measured by the Landsat 8 satellite on the seventh of June 2026, and by NASA's MODIS sensor on the tenth of June.", None),
        ("A true-colour photograph comes from the European Sentinel-2 satellite.", None),
        ("Air temperature, sunshine, and the height to which air can mix come from the ERA5 weather record.", None),
        ("Smog, measured as PM2.5, comes from the European CAMS air-quality model. It is an estimate, not a ground station.", None),
        ("Sun and shade, hours of sunlight, and open sky are calculated in the app from building heights and the position of the sun.", None),
    ]},
    {"img": "heat_walled", "chapter": "Ground heat measured from space", "lines": [
        ("This is the Walled City: a square of one point two by one point two kilometres around Rang Mahal.", "map"),
        ("These colours are not drawn by us. They are the ground temperature measured by Landsat 8 at ten thirty-five in the morning on the seventh of June 2026.", "map"),
        ("Landsat measures heat in squares of thirty metres. Blue squares were cooler, and red squares were hotter.", "map"),
        ("The hottest area is here in the north-east, around Gurdwara Janam Asthan, where rooftops are dense and there are almost no trees.", "hot"),
        ("This cooler area near Lohari Gate follows a green garden strip along the old city wall.", "cool"),
        ("Across the whole square, the typical ground temperature was fifty-one point six degrees Celsius, and the hottest spot reached fifty-three point nine.", "num"),
        ("This is not a land-use map. Red does not mean building, and blue does not mean road. Every colour is a measured temperature.", "map"),
    ]},
    {"img": "heat_dha", "chapter": "Walled City versus DHA Phase 2", "lines": [
        ("Now we switch to DHA Phase 2. Same satellite, same morning, same colour scale.", "map"),
        ("Here the typical ground temperature was fifty point one degrees: one and a half degrees cooler than the Walled City.", "typical"),
        ("The main reason is greenery. Trees and grass cover forty-six percent of DHA's open ground, but only five percent in the Walled City.", "green"),
        ("Buildings also cover fifty-four percent of the ground in the Walled City, but only twenty-eight percent in DHA.", "map"),
    ]},
    {"img": "photo", "chapter": "Checking with a satellite photo", "lines": [
        ("How do we know the heat map is right? We check it against a real photograph.", "full"),
        ("This layer drapes a true-colour Sentinel-2 photo, taken on the fifth of June 2026, over the same buildings.", "layer"),
        ("The green strip near Lohari Gate is exactly where the heat map showed cooler, blue squares.", "green"),
        ("And the hottest red squares match crowded rooftops with no trees. The photograph confirms the measurement.", "map"),
    ]},
    {"img": "sun", "chapter": "The sun, hour by hour", "lines": [
        ("Next, the sun. The app calculates the position of the sun over Lahore for every hour of the day.", "map"),
        ("Then, for every roof, wall and street, it checks whether a taller building blocks the sunlight.", "map"),
        ("This is five in the evening on the tenth of June, the hottest day of the year, when the air reached forty-four degrees.", "clock"),
        ("The sun is thirty-one degrees above the western horizon. Yellow surfaces are in direct sunlight; blue surfaces are in shadow.", "map"),
        ("Over the whole day, the Walled City's narrow streets received seven point eight hours of direct sun, while DHA's open streets received eleven point four.", "map"),
    ]},
    {"img": "building", "chapter": "One building, floor by floor", "lines": [
        ("GIS can go down to a single building. This is D-Plaza, near Rang Mahal, about twenty-six metres tall.", "bld"),
        ("The chart shows how many hours of direct sun the outer walls of each floor receive.", "chart"),
        ("The lowest walls get only four point two hours of sun, while the top floor gets seven hours.", "chart"),
        ("Landsat measured fifty-one point four degrees on its roof. Each floor of the same building lives in a different climate.", "bld"),
    ]},
    {"img": "smog", "chapter": "Smog night", "lines": [
        ("Lahore's second problem is smog. This is the night of the tenth of November 2026, at ten o'clock.", "map"),
        ("PM2.5 reached two hundred and fifty-two micrograms per cubic metre. On the US air-quality scale, that is Hazardous.", "label"),
        ("The ERA5 weather record shows that the air could mix only up to twenty metres, about the height of the rooftops.", "mixing"),
        ("That creates a lid over the city. The smoke has nowhere to go, and it stays trapped in the streets.", "lid"),
        ("Narrow lanes are hit hardest. A typical Walled City street sees only fifty-three percent of the sky, compared with ninety-three percent in DHA.", "map"),
        ("Over the past year, PM2.5 stayed above the World Health Organization's daily limit on every single day.", "map"),
    ]},
    {"card": "close", "chapter": "Why GIS", "lines": [
        ("A weather report gives one number for the whole city. GIS shows where the heat and the smoke actually collect.", None),
        ("By combining satellites, weather records and building heights on one map, we can see which streets need shade, which roofs could be painted white, and which lanes trap smoke.", None),
        ("Our model estimates that white lime-washed roofs would be about thirteen degrees cooler in the afternoon, while the streets below barely change. Every surface needs its own solution.", None),
        ("All of this uses free, open data, and the same method can be applied to any city in Pakistan.", None),
        ("Heat has an address. GIS finds it. Thank you.", None),
    ]},
]
print(len(SCENES), "scenes,", sum(len(s["lines"]) for s in SCENES), "narration lines")

## Record the voice
Each narration line is recorded as its own clip, so every subtitle matches the voice exactly.

In [ ]:
LEAD, GAP, TAIL = 0.7, 0.35, 1.0      # seconds of silence: scene start, between lines, scene end
SR = 48000

def clip_path(si, li):
    return f"{WORK}/tts/s{si:02d}_l{li:02d}.wav"

def to_wav(src, dst):
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", src, "-ac", "1", "-ar", str(SR), dst], check=True)

def read_wav(path):
    with wave.open(path) as w:
        return np.frombuffer(w.readframes(w.getnframes()), dtype=np.int16)

def write_wav(path, data):
    with wave.open(path, "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(SR); w.writeframes(data.astype(np.int16).tobytes())

async def synth_all():
    if TTS_ENGINE == "edge":
        import edge_tts
    for si, sc in enumerate(SCENES):
        for li, (text, _) in enumerate(sc["lines"]):
            out = clip_path(si, li)
            if os.path.exists(out):
                continue
            if TTS_ENGINE == "silent":
                write_wav(out, np.zeros(int(SR * (len(text.split()) / 2.6 + 0.3)), np.int16))
                continue
            mp3 = out[:-4] + ".mp3"
            for v in [VOICE] + VOICE_FALLBACKS:
                try:
                    await edge_tts.Communicate(text, v, rate=RATE).save(mp3)
                    break
                except Exception as e:
                    print("voice", v, "failed:", e)
            to_wav(mp3, out)
        print("recorded scene", si + 1, "/", len(SCENES))

In [ ]:
await synth_all()

## Timing, scene audio and subtitles

In [ ]:
def build_timeline():
    """Scene audio files plus the start/end time of every line inside its scene."""
    plan = []
    for si, sc in enumerate(SCENES):
        parts, t, cues = [np.zeros(int(SR * LEAD), np.int16)], LEAD, []
        for li, (text, focus) in enumerate(sc["lines"]):
            a = read_wav(clip_path(si, li))
            cues.append({"text": text, "focus": focus, "t0": t, "t1": t + len(a) / SR})
            parts += [a, np.zeros(int(SR * GAP), np.int16)]
            t += len(a) / SR + GAP
        parts.append(np.zeros(int(SR * (TAIL - GAP)), np.int16))
        audio = np.concatenate(parts)
        write_wav(f"{WORK}/scenes/audio_{si:02d}.wav", audio)
        plan.append({"dur": len(audio) / SR, "cues": cues})
    return plan

PLAN = build_timeline()
total = sum(p["dur"] for p in PLAN)
print(f"video length: {int(total // 60)} min {total % 60:.0f} s")

def split_cue(text, t0, t1, max_chars=110):
    """Long lines are shown in parts, broken at punctuation where possible.
    Each part stays on screen for a share of the time proportional to its length."""
    clauses = re.split(r"(?<=[.,;:?])\s+", text)
    pieces = []
    for c in clauses:
        if len(c) <= max_chars:
            pieces.append(c)
        else:                                   # very long clause: cut into equal word groups
            words = c.split(); n = math.ceil(len(c) / max_chars); k = math.ceil(len(words) / n)
            pieces += [" ".join(words[i:i + k]) for i in range(0, len(words), k)]
    parts, cur = [], ""
    for pc in pieces:
        if cur and len(cur) + 1 + len(pc) > max_chars:
            parts.append(cur); cur = pc
        else:
            cur = (cur + " " + pc).strip()
    parts.append(cur)
    n = sum(len(p) for p in parts); out, t = [], t0
    for p in parts:
        d = (t1 - t0) * len(p) / n
        out.append((p, t, t + d)); t += d
    return out

def srt_time(t):
    ms = int(round(t * 1000)); h, ms = divmod(ms, 3600000); m, ms = divmod(ms, 60000); s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def write_srt():
    off, n, rows = 0.0, 1, []
    for p in PLAN:
        for c in p["cues"]:
            for text, a, b in split_cue(c["text"], c["t0"], c["t1"]):
                rows.append(f"{n}\n{srt_time(off + a)} --> {srt_time(off + b)}\n{text}\n"); n += 1
        off += p["dur"]
    open(OUT_SRT, "w", encoding="utf-8").write("\n".join(rows))
    print("wrote", OUT_SRT, "with", n - 1, "subtitles")

write_srt()

## Drawing: title cards, subtitles and the chapter label

In [ ]:
INK, PAPER, MUTED, ACCENT = (11, 14, 19), (236, 230, 218), (154, 163, 177), (238, 138, 69)

def wrap(draw, text, f, width):
    lines, cur = [], ""
    for w in text.split():
        t = (cur + " " + w).strip()
        if draw.textlength(t, font=f) <= width:
            cur = t
        else:
            lines.append(cur); cur = w
    lines.append(cur)
    return lines

def card(kind):
    im = Image.new("RGB", (W, H), INK); d = ImageDraw.Draw(im)
    # faint grid, like a map sheet
    for x in range(0, W, 120):
        d.line([(x, 0), (x, H)], fill=(20, 25, 33))
    for y in range(0, H, 120):
        d.line([(0, y), (W, y)], fill=(20, 25, 33))
    L = 160
    if kind == "title":
        d.text((L, 190), "GIS DAY 2026  ·  GIS: POWERING PURPOSE AND INNOVATION", font=font(30, "SemiBold"), fill=ACCENT)
        d.text((L, 260), "The Lahore Canyon", font=font(140, "Bold"), fill=PAPER)
        d.rectangle([L, 440, L + 120, 446], fill=ACCENT)
        d.text((L, 480), "Heat has an address. GIS finds it.", font=font(56, "Medium"), fill=ACCENT)
        d.text((L, 610), "Muhammad Mehdi  &  Hussain Ahmad", font=font(48, "SemiBold"), fill=PAPER)
        d.text((L, 678), "BS Geography · 7th Semester (Morning)", font=font(36), fill=MUTED)
        d.text((L, 725), "Institute of Geography, University of the Punjab, Lahore", font=font(36), fill=MUTED)
    elif kind == "data":
        d.text((L, 90), "DATA AND METHOD", font=font(30, "SemiBold"), fill=ACCENT)
        d.text((L, 135), "Where every number comes from", font=font(72, "Bold"), fill=PAPER)
        rows = [("Buildings", "Google Open Buildings 2.5D", "2023, 4 m", "SATELLITE-DERIVED", (110, 190, 145)),
                ("Ground temperature", "Landsat 8 and NASA MODIS", "7 & 10 Jun 2026", "MEASURED", (110, 190, 145)),
                ("Satellite photo", "Sentinel-2 (ESA Copernicus)", "5 Jun & 7 Nov 2026", "PHOTO", (110, 190, 145)),
                ("Air temperature, mixing height", "ERA5 weather record", "Nov 2025 – Nov 2026", "RECORD", (110, 190, 145)),
                ("PM2.5 smog", "CAMS air-quality model", "Nov 2025 – Nov 2026", "MODEL", (238, 138, 69)),
                ("Sun, shade, open sky", "Calculated in the app", "10 Jun & 10 Nov 2026", "CALCULATED", (255, 210, 122))]
        y = 280
        for a_, b_, c_, tag, col in rows:
            d.line([(L, y - 20), (W - L, y - 20)], fill=(40, 46, 58), width=2)
            d.text((L, y), a_, font=font(34, "SemiBold"), fill=PAPER)
            d.text((L + 600, y), b_, font=font(34), fill=PAPER)
            d.text((L + 1110, y + 3), c_, font=font(28), fill=MUTED)
            tf = font(22, "SemiBold"); tw = d.textlength(tag, font=tf)
            d.rounded_rectangle([W - L - tw - 28, y + 2, W - L, y + 38], radius=6, fill=tuple(int(v * 0.25) for v in col))
            d.text((W - L - tw - 14, y + 7), tag, font=tf, fill=col)
            y += 88
    elif kind == "close":
        d.text((L, 170), "WHY GIS", font=font(30, "SemiBold"), fill=ACCENT)
        d.text((L, 230), "Heat has an address.", font=font(120, "Bold"), fill=PAPER)
        d.text((L, 370), "GIS finds it.", font=font(120, "Bold"), fill=ACCENT)
        d.text((L, 580), "Muhammad Mehdi  &  Hussain Ahmad", font=font(44, "SemiBold"), fill=PAPER)
        d.text((L, 645), "Institute of Geography, University of the Punjab, Lahore", font=font(34), fill=MUTED)
        d.text((L, 710), "muhammad-mehdi12.github.io/lahore-canyon", font=font(34, "Medium"), fill=(185, 198, 236))
    return im

SUB_FONT = font(44, "SemiBold")
_sub_cache = {}

def subtitle_layer(text):
    """Transparent overlay with the subtitle text on a dark box near the bottom."""
    if text in _sub_cache:
        return _sub_cache[text]
    lay = Image.new("RGBA", (W, H), (0, 0, 0, 0)); d = ImageDraw.Draw(lay)
    lines = wrap(d, text, SUB_FONT, 1500)
    lh = 58; bw = max(d.textlength(l, font=SUB_FONT) for l in lines) + 64; bh = lh * len(lines) + 36
    x0, y0 = (W - bw) / 2, H - 70 - bh
    d.rounded_rectangle([x0, y0, x0 + bw, y0 + bh], radius=14, fill=(8, 10, 14, 200))
    for i, l in enumerate(lines):
        tw = d.textlength(l, font=SUB_FONT)
        d.text(((W - tw) / 2, y0 + 16 + i * lh), l, font=SUB_FONT, fill=(255, 255, 255, 255))
    _sub_cache[text] = lay
    return lay

def chapter_layer(n, title):
    lay = Image.new("RGBA", (W, H), (0, 0, 0, 0)); d = ImageDraw.Draw(lay)
    f1, f2 = font(26, "SemiBold"), font(34, "SemiBold")
    num = f"{n:02d}"
    w = d.textlength(num, font=f1) + d.textlength(title, font=f2) + 90
    d.rounded_rectangle([40, 36, 40 + w, 100], radius=12, fill=(8, 10, 14, 215))
    d.text((64, 52), num, font=f1, fill=ACCENT + (255,))
    d.text((64 + d.textlength(num, font=f1) + 26, 47), title, font=f2, fill=PAPER + (255,))
    return lay

## Camera moves and rendering
The camera eases smoothly to each line's focus point. Each scene is encoded to its own file, then all scenes are joined.

In [ ]:
def ease(t):
    t = min(max(t, 0.0), 1.0)
    return t * t * (3 - 2 * t)

MOVE = 1.6   # seconds for each camera move

def view_at(t, cues, presets, scale):
    """Camera box at time t: starts on the full view, then eases to each line's focus as that line begins."""
    cur = presets.get("full", (960, 540, 1920))
    for c in cues:
        start = max(c["t0"] - 0.3, 0.0)
        if t < start:
            break
        target = presets.get(c["focus"], cur) if c["focus"] else cur
        k = ease((t - start) / MOVE)
        cur = tuple(a + (b - a) * k for a, b in zip(cur, target))
    cx, cy, vw = (v * scale for v in cur)
    vh = vw * H / W
    SW, SH = 1920 * scale, 1080 * scale
    cx = min(max(cx, vw / 2), SW - vw / 2); cy = min(max(cy, vh / 2), SH - vh / 2)
    return (cx - vw / 2, cy - vh / 2, cx + vw / 2, cy + vh / 2)

def render_scene(si):
    sc, plan = SCENES[si], PLAN[si]
    dur = plan["dur"]; nfr = int(round(dur * FPS))
    src = card(sc["card"]) if "card" in sc else Image.open(f"{WORK}/img/{sc['img']}.jpg").convert("RGB")
    scale = src.width / 1920
    presets = FOCUS.get(sc.get("img"), {})
    chap = chapter_layer(si + 1, sc["chapter"]) if "img" in sc else None
    subs = [s for c in plan["cues"] for s in split_cue(c["text"], c["t0"], c["t1"])]
    tmp = f"{WORK}/scenes/scene_{si:02d}.tmp.mp4"; final = f"{WORK}/scenes/scene_{si:02d}.mp4"
    cmd = ["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{W}x{H}", "-r", str(FPS), "-i", "-",
           "-i", f"{WORK}/scenes/audio_{si:02d}.wav", "-c:v", "libx264", "-preset", "veryfast", "-crf", "18", "-pix_fmt", "yuv420p",
           "-c:a", "aac", "-b:a", "192k", "-ar", "48000", "-shortest", "-movflags", "+faststart", tmp]
    p = subprocess.Popen(cmd, stdin=subprocess.PIPE)
    last_box, base = None, None
    for f in range(nfr):
        t = f / FPS
        box = view_at(t, plan["cues"], presets, scale) if "img" in sc else (0, 0, W, H)
        if box != last_box:
            base = src.resize((W, H), Image.BICUBIC, box=box, reducing_gap=2.0) if "img" in sc else src
            last_box = box
        fr = base.convert("RGBA")
        if chap:
            fr.alpha_composite(chap)
        cur = next((s for s in subs if s[1] <= t < s[2] + 0.15), None)
        if cur:
            fr.alpha_composite(subtitle_layer(cur[0]))
        fr = fr.convert("RGB")
        fade = min(t / 0.5, (dur - t) / 0.5, 1.0)
        if fade < 1:
            fr = Image.blend(Image.new("RGB", (W, H)), fr, max(fade, 0))
        p.stdin.write(fr.tobytes())
    p.stdin.close(); p.wait()
    if p.returncode != 0:
        raise RuntimeError(f"ffmpeg failed on scene {si}")
    os.replace(tmp, final)
    return final

In [ ]:
files = []
for si in range(len(SCENES)):
    files.append(render_scene(si))
    print(f"scene {si + 1}/{len(SCENES)} done: {SCENES[si]['chapter']}")

with open(f"{WORK}/scenes/list.txt", "w") as fh:
    for f in files:
        fh.write(f"file '{os.path.abspath(f)}'\n")
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-f", "concat", "-safe", "0", "-i", f"{WORK}/scenes/list.txt",
                "-c", "copy", "-movflags", "+faststart", OUT_MP4], check=True)
print("finished:", OUT_MP4, f"({os.path.getsize(OUT_MP4) / 1e6:.0f} MB)")

## Watch and download

In [ ]:
from IPython.display import HTML
from base64 import b64encode
small = f"{WORK}/preview.mp4"
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", OUT_MP4, "-vf", "scale=960:-2", "-c:v", "libx264", "-crf", "28", "-preset", "veryfast", "-c:a", "aac", "-b:a", "96k", small], check=True)
HTML(f'<video width="960" controls src="data:video/mp4;base64,{b64encode(open(small, "rb").read()).decode()}"></video>')

In [ ]:
try:
    from google.colab import files as colab_files
    colab_files.download(OUT_MP4)
    colab_files.download(OUT_SRT)
except ImportError:
    print("Not running in Colab: the files are in", os.getcwd())